# Excitatory-Inhibitory RNN

Reproduce the E-I RNN tutorial from Song, Yang & Wang (2016):
train an **Excitatory-Inhibitory RNN** on neurogym's PerceptualDecisionMaking task,
then perform **selectivity analysis**, **connectivity visualization**, **PCA**, and **fixed-point analysis**.

Reference:
> Song, H.F., Yang, G.R. and Wang, X.J., 2016.
> Training excitatory-inhibitory recurrent neural networks for cognitive tasks:
> a simple and flexible framework. PLoS computational biology, 12(2).

**Recipe**: model=`ei_rnn`, dataset=`perceptual_decision_making`, objective=`supervised`,
analysis=`fixed_points` + `dimensionality` + `linearization`.

## 0. Setup

In [ ]:
import sys
sys.path.append('../src')

import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from neuralrnn import AutoConfig, AutoModel, Trainer, TrainingArguments
from neuralrnn import SupervisedObjective, load_dataset
from neuralrnn.analysis import fit_pca, find_fixed_points, linearize, dominant_direction

torch.manual_seed(42)
np.random.seed(42)

## 1. Task data (neurogym)

We use the **PerceptualDecisionMaking** task from neurogym.
The network receives two noisy inputs (evidence for choice 1 and choice 2)
and must decide which is larger.

In [ ]:
# Load dataset with variable timing (matches reference implementation)
timing = {
    'fixation': ('choice', (50, 100, 200, 400)),
    'stimulus': ('choice', (100, 200, 400, 800)),
}
ds = load_dataset('perceptual_decision_making', batch_size=16, seq_len=100,
                  dt=20, timing=timing)
print('input_dim =', ds.input_dim, '| n_actions =', ds.output_dim)

In [ ]:
# Visualize sample trials (trial-aligned dataset interface)
n_show = 3
ds_viz = load_dataset('perceptual_decision_making', batch_size=16, n_trials=n_show,
                      dt=20, timing=timing)
colors = plt.cm.Set1(np.linspace(0, 1, n_show))

fig, axes = plt.subplots(ds.input_dim + 1, 1, figsize=(8, 6), sharex=True)
trial_labels = []
for i in range(n_show):
    cond = ds_viz.conditions[i]
    n = cond['n_steps']
    ob = ds_viz.inputs[i, :n].numpy()
    gt = ds_viz.targets[i, :n].numpy()
    t = np.arange(n) * ds_viz.dt / 1000
    for ch in range(ds.input_dim):
        axes[ch].plot(t, ob[:, ch], color=colors[i], alpha=0.8, lw=1.2)
    axes[-1].plot(t, gt, color=colors[i], alpha=0.8, lw=1.2)
    trial_labels.append(f"trial {i}: gt={cond.get('ground_truth', '?')}, coh={cond.get('coh', 0)}")

channel_names = ['Fixation', 'Stim-L', 'Stim-R']
for ch in range(ds.input_dim):
    axes[ch].set_ylabel(channel_names[ch])
axes[-1].set_ylabel('Target')
axes[-1].set_xlabel('Time (s)')
fig.legend(trial_labels, loc='upper right', fontsize=8)
fig.suptitle('Sample trials: inputs and target output', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Define and train E-I RNN

The E-I RNN enforces **Dale's principle**: excitatory units have non-negative
outgoing weights, inhibitory units have non-positive outgoing weights.
Readout is restricted to excitatory units only (long-range projections are
exclusively excitatory in the mammalian cortex).

In [ ]:
# Create E-I RNN with 50 units (40 excitatory, 10 inhibitory)
hidden_size = 50
cfg = AutoConfig.for_model('ei_rnn',
                           input_dim=ds.input_dim,
                           latent_dim=hidden_size,
                           output_dim=ds.output_dim,
                           dt=ds.dt,
                           sigma_rec=0.15,
                           nonlinearity_mode='post_blend')  # Match reference: f((1-α)z + α·pre)
model = AutoModel.from_config(cfg)
print(model)
print(f'\nExcitatory units: {model.e_size}, Inhibitory units: {model.i_size}')
print(f'Readout from E units only: {cfg.readout_e_only}')
print(f'Total parameters: {model.num_parameters()}')

In [ ]:
# Train using the unified Trainer
objective = SupervisedObjective(task_type='classification')
history = Trainer(model, ds, objective,
                  TrainingArguments(max_steps=5000, learning_rate=1e-3,
                                   log_every=500)).train()

## 3. Run post-training and collect neural activity

Run the trained model on many trials with fixed timing to collect
neural activity for analysis.

In [ ]:
out.states.shape

In [ ]:
# Collect activity with fixed timing (constant 500 ms fixation/stimulus) for analysis.
# Trial-aligned dataset: whole pre-generated trials + per-trial conditions with epoch bounds.
ds_analysis = load_dataset('perceptual_decision_making', batch_size=16, n_trials=500, dt=20,
                           timing={'fixation': ('constant', 500),
                                   'stimulus': ('constant', 500)})

num_trial = len(ds_analysis)
activity_dict = {}
trial_infos = {}
stim_activity = [[], []]  # response for ground-truth 0 and 1

model.eval()
with torch.no_grad():
    out = model(ds_analysis.inputs)   # batched forward over all trials (already batch-first)
states_all = out.states.numpy()       # (N, T, latent_dim)
outputs_all = out.outputs.numpy()     # (N, T, output_dim)

for i, cond in enumerate(ds_analysis.conditions):
    n = cond['n_steps']               # true length of this trial (unpadded)
    rnn_activity = states_all[i, :n]
    activity_dict[i] = rnn_activity

    # Compute performance from the last step of the trial
    choice = int(np.argmax(outputs_all[i, n - 1]))
    correct = bool(choice == int(ds_analysis.targets[i, n - 1]))
    trial_infos[i] = {**cond, 'correct': correct, 'choice': choice}

    # Compute stimulus selectivity (epoch bounds from conditions)
    s, e = cond['epochs']['stimulus']
    stim_activity[cond['ground_truth']].append(rnn_activity[s:e])

acc = np.mean([val['correct'] for val in trial_infos.values()])
print(f'Average performance: {acc:.3f}')

## 4. Plot neural activity from sample trials

In [ ]:
e_size = model.e_size
trial = 2

plt.figure(figsize=(8, 4))
plt.plot(activity_dict[trial][:, :e_size], color='red', alpha=0.5, label='Excitatory')
plt.plot(activity_dict[trial][:, e_size:], color='blue', alpha=0.5, label='Inhibitory')
plt.xlabel('Time step')
plt.ylabel('Activity')
plt.title(f'Neural activity (trial {trial})')
# plt.legend()
plt.show()

## 5. Compute stimulus selectivity ($d'$)

For each neuron, compute the stimulus period selectivity:
$$d' = \frac{\mu_1 - \mu_2}{\sqrt{(\sigma_1^2 + \sigma_2^2)/2}}$$
where $\mu_1, \sigma_1^2$ are the mean and variance of activity on choice-1 trials.

In [ ]:
mean_activity = []
std_activity = []
for ground_truth in [0, 1]:
    activity = np.concatenate(stim_activity[ground_truth], axis=0)
    mean_activity.append(np.mean(activity, axis=0))
    std_activity.append(np.std(activity, axis=0))

# Compute d'
selectivity = (mean_activity[0] - mean_activity[1])
selectivity /= np.sqrt((std_activity[0]**2 + std_activity[1]**2 + 1e-7) / 2)

# Sort index for selectivity, separately for E and I
ind_sort = np.concatenate((np.argsort(selectivity[:e_size]),
                           np.argsort(selectivity[e_size:]) + e_size))

# Plot distribution of stimulus selectivity
plt.figure(figsize=(6, 3))
plt.hist(selectivity[:e_size], bins=20, alpha=0.7, color='red', label='Excitatory')
plt.hist(selectivity[e_size:], bins=10, alpha=0.7, color='blue', label='Inhibitory')
plt.xlabel("Selectivity (d')")
plt.ylabel('Number of neurons')
plt.legend()
plt.title('Stimulus selectivity distribution')
plt.show()

## 6. Plot network connectivity sorted by stimulus selectivity

Visualize the effective recurrent weight matrix $W^{\text{rec}}$ sorted by
each neuron's selectivity index. This reveals the cluster structure
that emerges from training.

In [ ]:
# Get effective recurrent weight (with Dale's constraint applied)
W = model._recurrent_weight().detach().numpy()

# Sort by selectivity
W_sorted = W[:, ind_sort][ind_sort, :]
wlim = np.max(np.abs(W_sorted))

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(W_sorted, cmap='bwr_r', vmin=-wlim, vmax=wlim,
               extent=(-0.5, hidden_size-0.5, -0.5, hidden_size-0.5),
               interpolation='nearest')
plt.colorbar(im, ax=ax, label='Connection weight')
ax.set_xlabel('From neurons (sorted by d\' )')
ax.set_ylabel('To neurons (sorted by d\' )')
ax.set_title('Network connectivity (E-I RNN)')

# Mark E/I boundary
ax.axhline(y=e_size - 0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=e_size - 0.5, color='gray', linestyle='--', alpha=0.5)
ax.text(e_size/2, -2, 'E', ha='center', fontsize=10, color='red')
ax.text(e_size + model.i_size/2, -2, 'I', ha='center', fontsize=10, color='blue')

plt.tight_layout()
plt.show()

## 7. PCA analysis

Project neural activity onto the first two principal components
to visualize the low-dimensional dynamics.

In [ ]:
# Concatenate all activity for PCA fitting
activity_all = np.concatenate([activity_dict[i] for i in range(num_trial)], axis=0)
print(f'Activity shape for PCA: {activity_all.shape}')

pca = fit_pca(activity_all, n_components=2)
print('Explained variance ratio:', np.round(pca.explained_variance_ratio, 3))

In [ ]:
# Build DataFrame for trial info
df_trials = pd.DataFrame(trial_infos).T

# Publication-quality PCA visualization
colors_rgb = np.array([[27, 158, 119], [117, 112, 179]]) / 255.  # green, purple

cohs = sorted(df_trials['coh'].unique())
cohs_pos = [c for c in cohs if c > 0]
color_intensity = [0.4, 0.7, 1.0, 1.3][:len(cohs_pos)]

fig = plt.figure(figsize=(4, 4))
ax = fig.add_axes([0.2, 0.2, 0.6, 0.6])

for ground_truth in [0, 1]:
    if ground_truth == 0:
        cohs_ = cohs_pos[::-1]
        color_intensity_ = color_intensity[::-1]
    else:
        cohs_ = cohs_pos
        color_intensity_ = color_intensity
    for i_coh, coh in enumerate(cohs_):
        trials_idx = df_trials[(df_trials['coh'] == coh) &
                               (df_trials['ground_truth'] == ground_truth)].index
        if len(trials_idx) == 0:
            continue
        activity_avg = np.mean([activity_dict[i] for i in trials_idx], axis=0)
        activity_pc = pca.transform(activity_avg)
        color = colors_rgb[ground_truth] * color_intensity_[i_coh]
        signed_coh = coh * (2 * ground_truth - 1)
        ax.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-',
                color=color, ms=3, markeredgecolor='none', lw=1,
                label=f'{signed_coh:+.1f}')

ax.legend(title='Signed Coh', loc='upper left', bbox_to_anchor=(1.0, 1.0),
          frameon=False, fontsize=7)
ax.set_xlabel('PC 1', fontsize=7)
ax.set_ylabel('PC 2', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)
plt.title('E-I RNN: PCA of neural dynamics')
plt.show()

## 8. Fixed-point analysis

Search for approximate fixed points under the **0-coherence stimulus condition**
(fixation=1, left-stim=0.5, right-stim=0.5), then project onto the PCA plane
and draw the dominant eigenvector direction.

In [ ]:
# 0-coherence stimulus input
task_input = torch.tensor([1, 0.5, 0.5], dtype=torch.float32)

# Fixed-point search
fps = find_fixed_points(model, backend='numeric', task_input=task_input,
                        n_candidates=128, n_iters=10000, speed_tol=0.5)
print(f'Found {len(fps)} fixed points')

In [ ]:
# Visualize fixed points in PCA space
coords = pca.transform(fps.coords()) if len(fps) else np.empty((0, 2))

fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.15, 0.15, 0.75, 0.75])

# Overlay per-trial trajectories colored by ground truth
for i in range(min(100, num_trial)):
    activity_pc = pca.transform(activity_dict[i])
    color = colors_rgb[0] if trial_infos[i]['ground_truth'] == 0 else colors_rgb[1]
    ax.plot(activity_pc[:, 0], activity_pc[:, 1], '.', color=color, alpha=0.5, ms=1)

if len(fps):
    ax.plot(coords[:, 0], coords[:, 1], 'kx', ms=5, mew=0.5, label='Fixed points')
    # Dominant eigenvector direction at first fixed point
    lin = linearize(model, fps.points[0].z, task_input=task_input)
    d = dominant_direction(lin)
    seg = pca.transform(np.array([fps.points[0].z + 2 * d,
                                   fps.points[0].z - 2 * d]))
    ax.plot(seg[:, 0], seg[:, 1], 'r-', lw=2, label='Dominant eigenvector')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title('E-I RNN: Fixed points in PCA space')
plt.show()

## 9. Jacobian eigenvalue analysis

Compute the Jacobian at a fixed point to assess stability.
Eigenvalues inside the unit circle indicate stable directions.

In [ ]:
if len(fps) > 0:
    # Choose a fixed point near the center
    fp_coords = fps.coords()
    i_fp = np.argsort(fp_coords[:, 0])[len(fp_coords) // 2]
    
    # Linearize
    lin = linearize(model, fps.points[i_fp].z, task_input=task_input)
    eigvals = lin.eigenvalues
    
    # Plot eigenvalues in complex plane
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(np.real(eigvals), np.imag(eigvals), s=30, alpha=0.7)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Unit circle
    theta = np.linspace(0, 2 * np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, lw=0.5)
    
    ax.set_xlabel('Real')
    ax.set_ylabel('Imaginary')
    ax.set_title('Jacobian eigenvalues at fixed point')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()
    
    n_stable = np.sum(np.abs(eigvals) < 1)
    print(f'Stable eigenvalues (|λ|<1): {n_stable}/{len(eigvals)}')
    
    # Print top eigenvalues
    sorted_idx = np.argsort(-np.abs(eigvals))
    print('\nTop 5 eigenvalues by magnitude:')
    for idx in sorted_idx[:5]:
        print(f'  λ = {eigvals[idx]:.4f}, |λ| = {np.abs(eigvals[idx]):.4f}')
else:
    print('No fixed points found.')

---

## Summary

This tutorial demonstrated how to use the NeuralRNN framework to:

1. **Create an E-I RNN** with Dale's principle using `AutoConfig.for_model('ei_rnn', ...)`
2. **Train** on a perceptual decision-making task using the unified `Trainer`
3. **Analyze selectivity** ($d'$) of individual neurons
4. **Visualize connectivity** structure that emerges from training
5. **Perform PCA** to visualize low-dimensional dynamics
6. **Search for fixed points** and analyze their stability via Jacobian eigenvalues

All analysis tools (`fit_pca`, `find_fixed_points`, `linearize`, `dominant_direction`)
are model-agnostic and work with any model implementing the `recurrence`/`readout` interface.

### Key differences from vanilla CTRNN
- **Dale's principle**: E units have non-negative outgoing weights, I units non-positive
- **Readout from E units only**: matches biological long-range excitatory projections
- **EI initialization**: E weights scaled by I/E ratio for balanced excitation/inhibition

### Reference
Song, H.F., Yang, G.R. and Wang, X.J., 2016. Training excitatory-inhibitory recurrent
neural networks for cognitive tasks: a simple and flexible framework.
PLoS computational biology, 12(2).